# S04 — Monitoring, Optimization and Governance

**Time: about 65 minutes.**
Covers: Monitoring and Optimizing Streams · Governance and Security for Streaming Data

### What you will be able to do afterwards

- Read the six numbers that tell you whether a stream is healthy.
- Persist those numbers so you can alert on them.
- Tune micro-batch size against a measurement rather than a guess.
- Apply Unity Catalog governance to tables that are being written continuously.
- Leave nothing running.

### One note on listeners

`StreamingQueryListener` is not reliably available on serverless compute, so this notebook
reads `query.recentProgress` instead. Same numbers, no callback registration. If your
production streams run on classic compute, a listener is the better home for Step 1's logic
— the exercise is the same either way.

In [ ]:
from pyspark.sql import DataFrame, functions as F
from helpers import utils, event_stream, streaming_utils

cfg = utils.get_configs()
catalog, gold_schema = cfg["catalog"], cfg["schema_gold"]

metrics_table = utils.get_configs("stream_metrics")["table_gold"]
tuning_table = utils.get_configs("stream_tuning")["table_gold"]
events_silver = utils.get_configs("web_events")["table_silver"]
enriched_table = utils.get_configs("events_enriched")["table_silver"]
replay_path = event_stream.replay_path()

spark = utils.spark
print(metrics_table, tuning_table, sep="\n")

## Step 1 — Persist the progress metrics

**TO DO**

1. Start a stream over the replay directory (Auto Loader, `maxFilesPerTrigger = 2`,
   `processingTime` trigger) writing to a scratch table. Use a **fresh** checkpoint.
2. Let several micro-batches complete with `streaming_utils.await_batches`.
3. Use `streaming_utils.progress_summary(query)` to get one dict per batch.
4. Write those to `stream_metrics` in gold, with at least `batch_id`, `input_rows`,
   `rows_per_second`, `batch_duration_ms`, plus the query name and a run timestamp.
5. Stop the query.

**The six numbers, and what each one tells you**

| Metric | Reading it |
| :-- | :-- |
| `numInputRows` | how much arrived this batch |
| `processedRowsPerSecond` | throughput |
| `batchDuration` | if this exceeds your trigger interval, you are falling behind |
| `durationMs.triggerExecution` | where the time actually went |
| `eventTime.watermark` | how far event time has advanced |
| `stateOperators[*].numRowsTotal` | state size — the one that kills jobs slowly |

> **Question:** which single metric would you alert on first, and what threshold? Justify
> the number, don't just pick one.

In [ ]:
# TO DO: start a metered stream over the replay files


# TO DO: await batches, capture progress_summary, persist to stream_metrics, stop

## Step 2 — Tune the batch size against a measurement

**TO DO**

Run the *same* replay through Auto Loader twice, into two scratch tables with two fresh
checkpoints:

- Run A: `cloudFiles.maxFilesPerTrigger = 1`
- Run B: `cloudFiles.maxFilesPerTrigger = 6`

For each run record, in `stream_tuning` in gold: `config_name`, `max_files_per_trigger`,
`num_batches`, `total_rows`, `total_duration_ms`, `avg_batch_duration_ms`,
`rows_per_second`.

Then compare.

**What you should find.** Run B does fewer, larger batches. Total throughput usually
improves — fixed per-batch overhead is amortised over more rows. Per-batch latency gets
worse. That is the entire trade, and there is no setting that wins both.

> **Questions:**
> - Which run would you ship for a dashboard refreshed hourly? For a fraud check that must
>   fire within ten seconds?
> - `maxFilesPerTrigger` bounds files. `maxBytesPerTrigger` bounds size. When does the
>   difference between them matter?
> - Both runs processed identical data. Did the total row count match? If not, you have a
>   bug — find it before moving on.

In [ ]:
def timed_replay_run(config_name: str, max_files: int) -> dict:
    """
    Run the full replay through Auto Loader with a given batch size and return the metrics.

    Use a fresh checkpoint, schema location and target table per run, or run B will find
    nothing left to consume.
    """
    # TO DO
    pass


# TO DO: run both configs, persist to stream_tuning, compare

## Step 3 — Govern the streaming tables

A table being written continuously is still a table. It needs an owner, a description, and
its sensitive columns marked.

**TO DO**

1. Add a `COMMENT` to your silver events table and to `events_enriched`, saying what the
   table holds, its grain, and who owns it.
2. Tag the PII columns on `events_enriched` — `user_name` at minimum, plus anything else
   you consider personal:

   ```sql
   ALTER TABLE <table> ALTER COLUMN user_name SET TAGS ('pii' = 'true')
   ```

3. Query `information_schema.column_tags` to confirm the tags landed.
4. Run `SHOW GRANTS` on your gold schema and note who currently has access.

> **Questions:**
> - Tags are metadata; they enforce nothing by themselves. What makes a tag useful, and what
>   would you build on top of these?
> - Column lineage is captured automatically. Open the catalog explorer and look at the
>   lineage for `events_enriched`. What does it show that your notebook code does not?

In [ ]:
# TO DO: add table comments


# TO DO: tag the PII columns and verify via information_schema.column_tags


# TO DO: inspect the grants on your gold schema

## Step 4 — A governed view over a live table

**TO DO**

Create `v_events_masked` in gold over `events_enriched`:

- `user_name` masked unless the caller is in `pii_readers`, via `is_account_group_member()`.
- `session_id` truncated to its first 8 characters for everyone. It is a tracking
  identifier, and nobody analysing a funnel needs the whole thing.
- Everything else passed through.

Then query it and check what you can see.

> **Questions:**
> - The underlying table is being appended to continuously. Does the view see new rows
>   immediately? What would change if you had used a materialized view?
> - You truncated `session_id` rather than hashing or dropping it. What does each option
>   cost the analyst, and what does each protect?
> - Somebody needs to debug a specific user's session end to end. How do they get access,
>   and what should that leave behind?

In [ ]:
# TO DO: create the masked view


# TO DO: query it

## Step 5 — Shut everything down

**TO DO**

1. `streaming_utils.stop_all_streams()`.
2. Confirm `spark.streams.active` is empty.
3. List the checkpoint volume. Every scratch run left a directory behind.
4. Remove the checkpoints for the scratch tables from Steps 1 and 2 with
   `utils.reset_path(...)`, and drop the scratch tables. Keep the checkpoints for the
   tables the checks read.

> **Question:** an orphaned checkpoint is small, so why bother? Give the failure mode you
> are preventing — think about what happens when somebody reuses the table name in six
> months.

In [ ]:
streaming_utils.stop_all_streams()
print(f"Active queries: {len(spark.streams.active)}")

# TO DO: list the checkpoint volume and clean up the scratch runs

## Checks

In [ ]:
from helpers import test_runner

test_runner.run("S04-monitoring-and-governance")

## Recap

- Metrics you only print are metrics you cannot alert on. Persist them.
- `batchDuration` against your trigger interval is the single clearest "am I falling behind"
  signal.
- Batch size trades throughput against latency. Pick against a measurement and a
  requirement, never a default.
- Streaming tables need the same governance as batch tables. Being live is not an exemption.
- Stop your queries. On serverless, a forgotten `processingTime` trigger is a bill.

## Wrap-up questions

Answer these in your submission.

1. Explain exactly-once to somebody who has just learned that `foreachBatch` is
   at-least-once. Where does each guarantee come from?
2. A stream that ran fine for three weeks starts falling behind. Give your diagnostic order
   and the metric you check at each step.
3. You must reprocess the last 30 days after a logic bug. Describe the procedure — what you
   do with the checkpoint, the target table, and any downstream consumers.
4. Argue for and against replacing your Auto Loader ingestion with a real broker. Be
   specific about what would have to be true for the broker to win.
5. A late event arrives 45 minutes after its event time and your watermark is 10 minutes.
   Trace what happens to it, and describe what you would build if that event mattered.